https://huggingface.co/openai/whisper-large-v3

In [1]:
!pip install --upgrade pip
!pip install --upgrade transformers datasets[audio] accelerate

  Using cached pip-26.0.1-py3-none-any.whl.metadata (4.7 kB)
Using cached pip-26.0.1-py3-none-any.whl (1.8 MB)
  Attempting uninstall: pip
    Found existing installation: pip 25.3
    Uninstalling pip-25.3:
      Successfully uninstalled pip-25.3
zsh:1: no matches found: datasets[audio]


In [5]:
import torch
from transformers import AutoModelForSpeechSeq2Seq, AutoProcessor, pipeline
from datasets import load_dataset


def resolve_runtime():
    """Choose the best available acceleration backend."""
    if torch.cuda.is_available():
        return {
            "label": "NVIDIA CUDA GPU",
            "model_device": "cuda:0",
            "pipeline_device": 0,
            "torch_dtype": torch.float16,
        }

    mps_available = hasattr(torch.backends, "mps") and torch.backends.mps.is_available()
    if mps_available:
        return {
            "label": "Apple Silicon Metal (MPS)",
            "model_device": "mps",
            "pipeline_device": torch.device("mps"),
            "torch_dtype": torch.float16,
        }

    return {
        "label": "CPU",
        "model_device": "cpu",
        "pipeline_device": -1,
        "torch_dtype": torch.float32,
    }


runtime = resolve_runtime()
print(f"Using backend: {runtime['label']}")

model_id = "openai/whisper-large-v3"

model = AutoModelForSpeechSeq2Seq.from_pretrained(
    model_id,
    torch_dtype=runtime["torch_dtype"],
    low_cpu_mem_usage=True,
    use_safetensors=True,
    )
model.to(runtime["model_device"])

processor = AutoProcessor.from_pretrained(model_id)

pipe = pipeline(
    "automatic-speech-recognition",
    model=model,
    tokenizer=processor.tokenizer,
    feature_extractor=processor.feature_extractor,
    torch_dtype=runtime["torch_dtype"],
    device=runtime["pipeline_device"],
)

dataset = load_dataset("distil-whisper/librispeech_long", "clean", split="validation")
sample = dataset[0]["audio"]

result = pipe(sample)
print(result["text"])

Using backend: Apple Silicon Metal (MPS)


Device set to use mps


ValueError: You have passed more than 3000 mel input features (> 30 seconds) which automatically enables long-form generation which requires the model to predict timestamp tokens. Please either pass `return_timestamps=True` or make sure to pass no more than 3000 mel input features.

In [ ]:
from pathlib import Path

audio_path = Path("audio.mp3")  # Replace with your recorded file path
if not audio_path.exists():
    raise FileNotFoundError(f"Audio file not found: {audio_path.resolve()}")

result = pipe(str(audio_path), return_timestamps=True)
print(result["text"])

# Optional: save transcript beside the audio file
transcript_path = audio_path.with_suffix(".txt")
transcript_path.write_text(result["text"], encoding="utf-8")
print(f"Transcript saved to: {transcript_path}")

In [ ]:
result = pipe(["audio_1.mp3", "audio_2.mp3"], batch_size=2)

In [ ]:
generate_kwargs = {
    "max_new_tokens": 448,
    "num_beams": 1,
    "condition_on_prev_tokens": False,
    "compression_ratio_threshold": 1.35,  # zlib compression ratio threshold (in token space)
    "temperature": (0.0, 0.2, 0.4, 0.6, 0.8, 1.0),
    "logprob_threshold": -1.0,
    "no_speech_threshold": 0.6,
    "return_timestamps": True,
}

result = pipe(sample, generate_kwargs=generate_kwargs)
